# Update selected sources
Choose recent updates or full history, then inspect the destination and source plan.
**Run All is read-only by default:** both downloads and database writes are disabled.

Recent mode supports Massachusetts monthly PDFs and New York weekly workbooks.
MA fetches index pages and downloads only the latest selected report months.
NY downloads the current statewide/operator workbooks, which contain fiscal-year sheets;
identical retained and ingested bytes skip parsing. Weeks remain weeks.

For analysis, open `90_consolidated_ggr.ipynb`; for a visible parser walkthrough, open `31_massachusetts_pdf_walkthrough.ipynb`.

In [ ]:
database_file = "data/staging/gaming_nationwide.sqlite"  # explicit write destination
selected_sources = [("MA", "online_sports_betting"), ("NY", "online_sports_betting")]
collection_mode = "recent"  # "recent" or explicitly "history"
recent_report_limit = 2  # latest discovered MA months; does not limit NY workbook history
include_ny_operators = True
force_reparse = False  # deliberate replay: use a separately reviewed candidate database
run_downloads = False
allow_database_writes = False
# selected_sources = None is allowed only for explicit full-history collection.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.collect import COLLECTORS, inventory_collector_order, run_all_collectors
from variant_gaming.recent import RECENT_SOURCES, collect_recent
from variant_gaming.storage import connect_readonly

database_path = ROOT / database_file
print(f"Download/write destination: {database_path}")

## 1. Check the plan and destination
Pairs are `(state_code, product)`. Unsupported recent modes are rejected before any download
or database write. Full history must be chosen explicitly; it can be expensive.

The default destination is the existing staging database. It is not automatically backed up.
For a deliberate parser replay, choose a separately prepared candidate database and review the
differences before any replacement. Writing the original `data/gaming.sqlite` remains a separate action.
Any changed source or database can invalidate the approvals used by notebooks 91–92.

In [ ]:
if collection_mode not in {"recent", "history"}:
    raise ValueError("Choose collection_mode='recent' or 'history'")
if collection_mode == "history" and force_reparse:
    raise ValueError("force_reparse is a recent-mode option; history already parses its downloads")
scheduled = inventory_collector_order(ROOT)
if selected_sources is not None:
    requested = {(state.upper(), product) for state, product in selected_sources}
    if not requested or requested - COLLECTORS.keys():
        raise ValueError("Choose nonempty, registered state/product pairs")
    scheduled = [row for row in scheduled if (row[1], row[2]) in requested]
if collection_mode == "recent":
    if selected_sources is None or requested - RECENT_SOURCES:
        raise ValueError("Recent mode supports only MA/NY online sports betting. Select history explicitly for other sources.")
    if not isinstance(recent_report_limit, int) or isinstance(recent_report_limit, bool) or recent_report_limit < 1:
        raise ValueError("recent_report_limit must be a positive integer")
plan = pd.DataFrame(scheduled, columns=["wave", "state_code", "vertical"])
plan["mode"] = collection_mode
display(plan)

## 2. Download, validate, and write
Enable **both** switches only when ready to download reports and write to the displayed destination.
Recent mode returns one row per report, including failures. `stored_rows` counts rows upserted in this
run, not net new observations. Zero stored rows with `previously_ingested` means a verified unchanged
report was skipped. A retained file with missing observations still gets parsed.
A successful ingestion records its row count in the selected database. Older versions
without this receipt are parsed once to verify completeness before future skips.

`bytes_changed` compares with stored versions of that report; it is unknown on a first capture.
Parsed date bounds describe observations, not publication dates. A workbook may cover several years.
`validation` shows `passed`, `previously_ingested`, `failed`, or `not_run`. Missing values are not zeros.

Changed hashes remain separate. The collector never chooses a winning revision. Inspect conflicts in
notebook 90 before using changed observations. `force_reparse=True` updates parsed values for identical
source bytes and therefore requires a deliberate reviewed replay; leave it false for routine refreshes.
The returned table is the run log in memory; successful-ingestion counts use one small
`recent_ingestions` table in the same SQLite database. There is no separate log service.

In [ ]:
summary = pd.DataFrame()
if run_downloads and allow_database_writes:
    if collection_mode == "recent":
        reports = []
        for item in plan.itertuples(index=False):
            reports.append(collect_recent(
                item.state_code, item.vertical, root=ROOT, db_path=database_path,
                report_limit=recent_report_limit, include_operators=include_ny_operators,
                force_reparse=force_reparse,
            ))
        summary = pd.concat(reports, ignore_index=True)
    else:
        summary = run_all_collectors(root=ROOT, db_path=database_path, selected=selected_sources)
    display(summary)
else:
    print("Collection disabled. Both run_downloads and allow_database_writes must be True.")

In [ ]:
if not summary.empty:
    if collection_mode == "recent":
        needs_attention = summary[~summary["validation"].isin(["passed", "previously_ingested"])]
        display(needs_attention[["state_code", "report", "download_status", "validation", "reason", "source_file"]])
        # Sources with new bytes need review even when parsing and reconciliation passed.
        display(summary[summary["bytes_changed"].eq(True)][
            ["state_code", "report", "period_start", "period_end", "source_file", "source_sha256"]])
    else:
        needs_attention = summary[
            summary["run_status"].ne("completed") | summary["coverage_status"].ne("ok")
        ]
        display(needs_attention)

## 3. Inspect what is already stored
These queries are read-only. A failed refresh can leave older observations available.
Recent runs do not replace full-history coverage claims: the coverage table below describes the
last history run, while the second table shows stored rows and source versions now. Neither raw row
counts nor minimum/maximum dates prove complete coverage; notebook 90 exposes missing periods and conflicts.

In [ ]:
if database_path.exists():
    connection = connect_readonly(database_path)
    try:
        coverage = pd.read_sql_query("SELECT * FROM source_coverage", connection)
        stored = pd.read_sql_query("""
            SELECT state_code, vertical, COUNT(*) AS stored_rows,
                   COUNT(DISTINCT source_sha256) AS source_versions,
                   MIN(period_start) AS first_period, MAX(period_end) AS last_period
            FROM gaming_results GROUP BY state_code, vertical
        """, connection)
    finally:
        connection.close()
    display(plan.merge(coverage, on=["state_code", "vertical"], how="left"))
    display(plan.merge(stored, on=["state_code", "vertical"], how="left"))
else:
    print("No local database yet. An explicitly enabled collection creates it.")